In [1]:
import ee
ee.Initialize()

In [2]:
# Área de interesse - ÁREA REDUZIDA (região de Chókwe, Moçambique)
#aoi = ee.Geometry.Rectangle([33.49, -24.69, 33.51, -24.6])

# Melhor opção para agricultura
aoi = ee.Geometry.Polygon([
    [
        [33.49, -24.69],
        [33.51, -24.69],
        [33.51, -24.60],
        [33.49, -24.60],
        [33.49, -24.69]
    ]
])

## **Verificando quantos meses realmente têm dados**

In [5]:
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi)

meses_com_dados = 0
years = [2018, 2019, 2020, 2021, 2022, 2023]  # 6 anos

total_meses_possiveis = len(years) * 12

print("Verificando a disponibilidade dos dados (2018-2023):")
print("=" * 70)
print(f"Área: Chókwe, Moçambique")
print(f"Período: 2018 a 2023 ({len(years)} anos, {total_meses_possiveis} meses possíveis)")
print(f"Filtro: Imagens com <30% de nuvens será aplicado no download, aqui mostramos todas disponíveis")
print("-" * 70)

for year in years:
    print(f"\n{year}:")
    anos_com_dados = 0
    
    for month in range(1, 13):
        start = f'{year}-{month:02d}-01'
        
        if month == 12:
            end = f'{year+1}-01-01'
        else:
            end = f'{year}-{month+1:02d}-01'
        
        # Contar imagens disponíveis (sem filtro de nuvem ainda)
        count = collection.filterDate(start, end).size().getInfo()
        
        if count > 0:
            meses_com_dados += 1
            anos_com_dados += 1
            
            # Mostrar detalhe do mês
            if count >= 10:
                status = "🟢 MUITAS"
            elif count >= 5:
                status = "🟡 BOAS"
            elif count >= 1:
                status = "🟠 POUCAS"
            
            print(f"{year}-{month:02d}: {count:2d} imagens disponíveis {status}")
        else:
            print(f"{year}-{month:02d}: SEM DADOS")
    
    # Resumo do ano
    if anos_com_dados > 0:
        print(f" Total {year}: {anos_com_dados}/12 meses com dados")

print("\n" + "=" * 70)
print(f" RESUMO FINAL:")
print(f"   • Total de meses com dados: {meses_com_dados} de {total_meses_possiveis} possíveis")
print(f"   • Total de arquivos TIFF esperados: {meses_com_dados}")
print(f"   • Cobertura: {(meses_com_dados/total_meses_possiveis*100):.1f}% do período")
print("=" * 70)

# Estatísticas adicionais
print("\n ESTIMATIVA DE ARMAZENAMENTO:")
print(f"   • Com scale=100m: ~{meses_com_dados * 0.05:.1f} MB total")
print(f"   • Com scale=30m:  ~{meses_com_dados * 0.2:.1f} MB total")
print(f"   • Com scale=10m:  ~{meses_com_dados * 0.8:.1f} MB total")

# Verificar cobertura por ano
print("\n COBERTURA POR ANO:")
for year in years:
    count_year = 0
    for month in range(1, 13):
        start = f'{year}-{month:02d}-01'
        end = f'{year}-{month+1:02d}-01' if month < 12 else f'{year+1}-01-01'
        count = collection.filterDate(start, end).size().getInfo()
        if count > 0:
            count_year += 1
    print(f"   {year}: {count_year}/12 meses ({count_year/12*100:.0f}%)")

VERIFICANDO DISPONIBILIDADE DE DADOS SENTINEL-2 (2018-2023):
Área: Chókwe, Moçambique
Período: 2018 a 2023 (6 anos, 72 meses possíveis)
Filtro: Imagens com <30% de nuvens será aplicado no download, aqui mostramos todas disponíveis
----------------------------------------------------------------------

2018:
2018-01:  1 imagens disponíveis 🟠 POUCAS
2018-02: SEM DADOS
2018-03: SEM DADOS
2018-04: SEM DADOS
2018-05:  1 imagens disponíveis 🟠 POUCAS
2018-06:  1 imagens disponíveis 🟠 POUCAS
2018-07: SEM DADOS
2018-08: SEM DADOS
2018-09:  3 imagens disponíveis 🟠 POUCAS
2018-10: SEM DADOS
2018-11:  1 imagens disponíveis 🟠 POUCAS
2018-12:  3 imagens disponíveis 🟠 POUCAS
  📊 Total 2018: 6/12 meses com dados

2019:
2019-01:  7 imagens disponíveis 🟡 BOAS
2019-02:  5 imagens disponíveis 🟡 BOAS
2019-03:  6 imagens disponíveis 🟡 BOAS
2019-04:  6 imagens disponíveis 🟡 BOAS
2019-05:  6 imagens disponíveis 🟡 BOAS
2019-06:  6 imagens disponíveis 🟡 BOAS
2019-07:  6 imagens disponíveis 🟡 BOAS
2019-08:  6 im

## **Baixando os dados de Satelite_sentinel_2**

In [6]:
print(f"Área aproximada: {aoi.area().getInfo() / 1e6:.2f} km²")

# Função NDVI
def add_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

# Dataset Copernicus Sentinel-2
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(aoi)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
              .map(add_ndvi)
             )

years = [2018, 2019, 2020, 2021, 2022, 2023]

print("Iniciando exportações mensais...")

for year in years:
    for month in range(1, 13):
        
        start_date = f'{year}-{month:02d}-01'
        
        if month == 12:
            end_date = f'{year+1}-01-01'
        else:
            end_date = f'{year}-{month+1:02d}-01'
        
        # Verificar se há imagens no período
        count = collection.filterDate(start_date, end_date).size().getInfo()
        
        if count > 0:
            ndvi_month = (collection
                          .filterDate(start_date, end_date)
                          .select('NDVI')
                          .mean()
                          .clip(aoi))
            
            # ESCALA REDUZIDA para arquivos menores
            # 100m = ~10x menos pixels que 10m, resultando em arquivos de poucos MB
            task = ee.batch.Export.image.toDrive(
                image=ndvi_month,
                description=f'NDVI_{year}_{month:02d}',
                folder='NDVI_Chokwe1',
                fileNamePrefix=f'NDVI_{year}_{month:02d}',
                scale=10,  # Pode-se alterar esse valor 
                region=aoi.getInfo()['coordinates'],
                maxPixels=1e13,
                fileFormat='GeoTIFF'
            )
            
            task.start()
            print(f"Iniciado: {year}-{month:02d} ({count} imagens) - Escala: 10m")
        else:
            print(f"Sem dados: {year}-{month:02d}")

print("\nTodas as exportações iniciadas!")
print("Monitore no Earth Engine: https://code.earthengine.google.com/tasks")

Área aproximada: 20.23 km²
Iniciando exportações mensais...
Iniciado: 2018-01 (1 imagens) - Escala: 10m
Sem dados: 2018-02
Sem dados: 2018-03
Sem dados: 2018-04
Iniciado: 2018-05 (1 imagens) - Escala: 10m
Iniciado: 2018-06 (1 imagens) - Escala: 10m
Sem dados: 2018-07
Sem dados: 2018-08
Iniciado: 2018-09 (3 imagens) - Escala: 10m
Sem dados: 2018-10
Iniciado: 2018-11 (1 imagens) - Escala: 10m
Iniciado: 2018-12 (3 imagens) - Escala: 10m
Iniciado: 2019-01 (1 imagens) - Escala: 10m
Iniciado: 2019-02 (2 imagens) - Escala: 10m
Iniciado: 2019-03 (1 imagens) - Escala: 10m
Iniciado: 2019-04 (1 imagens) - Escala: 10m
Iniciado: 2019-05 (5 imagens) - Escala: 10m
Iniciado: 2019-06 (5 imagens) - Escala: 10m
Iniciado: 2019-07 (5 imagens) - Escala: 10m
Iniciado: 2019-08 (4 imagens) - Escala: 10m
Iniciado: 2019-09 (4 imagens) - Escala: 10m
Iniciado: 2019-10 (4 imagens) - Escala: 10m
Iniciado: 2019-11 (2 imagens) - Escala: 10m
Iniciado: 2019-12 (3 imagens) - Escala: 10m
Iniciado: 2020-01 (4 imagens) - Es